# Network GAAE Runner

Parameterized GAAE pretraining notebook for any network combination.
Edit the `EXPERIMENT` dict below, then run all cells.

| `network_combo` | `data_version` | `n_rois` | `file_suffix` |
|---|---|---|---|
| `dmn` | `__v4__` | 46 | `_dmn_correlation_matrix_z_transformed.npz` |
| `hippo` | `__v5__` | 4 | `_hippocampus_correlation_matrix_z_transformed.npz` |
| `limbic` | `__v6__` | 12 | `_limbic_correlation_matrix_z_transformed.npz` |
| `dan` | `__v7__` | 26 | `_dorsal_attention_correlation_matrix_z_transformed.npz` |
| `dmn_hippo` | `__v8__` | 50 | `_dmn_hippo_correlation_matrix_z_transformed.npz` |
| `dmn_limbic` | `__v9__` | 58 | `_dmn_limbic_correlation_matrix_z_transformed.npz` |
| `dmn_limbic_hippo` | `__v10__` | 62 | `_dmn_limbic_hippo_correlation_matrix_z_transformed.npz` |
| `all_combined` | `__v11__` | 88 | `_all_combined_correlation_matrix_z_transformed.npz` |

In [ ]:
# ── EXPERIMENT CONFIG ─────────────────────────────────────────────────────────
EXPERIMENT = {
    "network_combo": "dmn_hippo",
    "data_version": "__v8__",
    "n_rois": 50,
    "file_suffix": "_dmn_hippo_correlation_matrix_z_transformed.npz",
    "knn_k": 8,
}
# ─────────────────────────────────────────────────────────────────────────────

REPO_ROOT = '/mnt/e/fyassine/ad-early-detection'
DATA_ROOT = f"{REPO_ROOT}/DATA/DELCODE/{EXPERIMENT['data_version']}/matrices"
METADATA_DIR = f"{REPO_ROOT}/DATA/DELCODE/__v3__/metadata"
GAAE_SPLITS_DIR = f"{METADATA_DIR}/splits_gaae"
COHORTS_CSV = f"{METADATA_DIR}/cohorts.csv"

CHECKPOINT_DIR = f"{REPO_ROOT}/CLASSIFIER/notebooks/checkpoints_gaae_{EXPERIMENT['network_combo']}"
WANDB_PROJECT = f"gaae-network-{EXPERIMENT['network_combo']}"

# Model hyperparams
IN_FEATURES = EXPERIMENT["n_rois"]
HIDDEN_DIM = max(IN_FEATURES, 64)
LATENT_DIM = 64
NUM_HEADS = 2
COND_DIM = 2
DROPOUT = 0.3
ADJ_LOSS_WEIGHT = 1.0

# Training
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 500
EARLY_STOPPING_PATIENCE = 50
SEED = 100
NUM_WORKERS = 0

print(f"Experiment: {EXPERIMENT['network_combo']}")
print(f"Data root: {DATA_ROOT}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"IN_FEATURES: {IN_FEATURES}, HIDDEN_DIM: {HIDDEN_DIM}, LATENT_DIM: {LATENT_DIM}")

## Imports

In [ ]:
import os
import sys
import json
import pickle
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import torch
from torch.utils.data import random_split, Subset
from torch_geometric.loader import DataLoader

classifier_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(classifier_root) not in sys.path:
    sys.path.insert(0, str(classifier_root))

from model.GAAE.models import GraphAttentionAutoencoderConditioned
from model.GAAE.dataset import GraphDatasetInMemoryFiltered
from model.GAAE.train import train_model_with_val_notebook_train_loss
from common.utils import knn_binary_adjacency_matrix_no_diag

sns.set_theme(style='whitegrid')
print('Imports OK')

## Setup

In [ ]:
try:
    wandb.login()
except Exception:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Seed: {SEED}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

## Dataset

In [ ]:
adjacency_args = {'k': EXPERIMENT['knn_k']}

# Use GAAE train+val CSVs from __v3__ metadata (subject IDs are the same across all versions)
gaae_train_csv = os.path.join(GAAE_SPLITS_DIR, 'train.csv')
gaae_val_csv = os.path.join(GAAE_SPLITS_DIR, 'val.csv')

# Combine train + val subject IDs for the full GAAE training pool,
# then split 85/15 internally (same as original GAAE notebook).
# If the CSV has Repseudonym, we build a temporary all-subjects CSV.
import tempfile
train_df = pd.read_csv(gaae_train_csv)
val_df = pd.read_csv(gaae_val_csv)
all_subjects_df = pd.concat([train_df, val_df], ignore_index=True).drop_duplicates(subset=['Repseudonym'])
tmp_all_csv = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False).name
all_subjects_df.to_csv(tmp_all_csv, index=False)

# Patient info CSV (age/sex)
patient_info_path = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False).name
if 'age' in all_subjects_df.columns and 'sex' in all_subjects_df.columns:
    pi_df = all_subjects_df[['Repseudonym', 'sex', 'age']].rename(columns={'Repseudonym': 'Repseudonym'})
    pi_df.to_csv(patient_info_path, index=False)
    patient_info_path_arg = patient_info_path
else:
    patient_info_path_arg = None

dataset = GraphDatasetInMemoryFiltered(
    root=DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    filter_csv_path=tmp_all_csv,
    patient_info_path=patient_info_path_arg,
    separator=',',
    file_suffix=EXPERIMENT['file_suffix'],
)

total_size = len(dataset)
train_size = int(0.85 * total_size)
val_size = total_size - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

print(f'Total subjects: {total_size}')
print(f'Train: {train_size} ({train_size / total_size * 100:.1f}%)')
print(f'Val: {val_size} ({val_size / total_size * 100:.1f}%)')
print(f'Node features (n_rois): {dataset[0].x.shape}')

dataset_info = {
    'network_combo': EXPERIMENT['network_combo'],
    'data_version': EXPERIMENT['data_version'],
    'n_rois': IN_FEATURES,
    'file_suffix': EXPERIMENT['file_suffix'],
    'knn_k': EXPERIMENT['knn_k'],
    'total_size': total_size,
    'train_size': train_size,
    'val_size': val_size,
    'batch_size': BATCH_SIZE,
}

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, worker_init_fn=worker_init_fn,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, worker_init_fn=worker_init_fn,
    pin_memory=torch.cuda.is_available(),
)

## Model

In [ ]:
model = GraphAttentionAutoencoderConditioned(
    in_features=IN_FEATURES,
    hidden_dim=HIDDEN_DIM,
    out_features=LATENT_DIM,
    cond_dim=COND_DIM,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
).to(device)

model_config = {
    'model_type': model.__class__.__name__,
    'in_features': IN_FEATURES,
    'hidden_size': HIDDEN_DIM,
    'latent_dim': LATENT_DIM,
    'attention_heads': NUM_HEADS,
    'cond_dim': COND_DIM,
    'dropout': DROPOUT,
    'device': device.type,
}

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Model config: {model_config}')

## Training

In [ ]:
best_model, history = train_model_with_val_notebook_train_loss(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    model_config=model_config,
    adj_loss_weight=ADJ_LOSS_WEIGHT,
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    dataset_info=dataset_info,
    project_name=WANDB_PROJECT,
)

run_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
try:
    wandb_run_name = wandb.run.name if wandb.run and wandb.run.name else 'gaae_run'
except Exception:
    wandb_run_name = 'gaae_run'
wandb_run_name = str(wandb_run_name).replace(' ', '-')
run_name = f'{wandb_run_name}_{run_timestamp}'

run_artifact_dir = os.path.join(CHECKPOINT_DIR, run_name)
os.makedirs(run_artifact_dir, exist_ok=True)

model_file = os.path.join(run_artifact_dir, f'model_{run_name}.pth')
torch.save(best_model, model_file)
print(f'Saved model: {model_file}')

config_to_save = {
    'run_name': run_name,
    'timestamp': run_timestamp,
    'experiment': EXPERIMENT,
    'dataset_info': dataset_info,
    'model_config': model_config,
    'training_config': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'adj_loss_weight': ADJ_LOSS_WEIGHT,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'seed': SEED,
    },
}
config_file = os.path.join(run_artifact_dir, 'run_config.json')
with open(config_file, 'w') as f:
    json.dump(config_to_save, f, indent=4, default=str)
print(f'Saved config: {config_file}')

## Loss Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title(f'GAAE Training — {EXPERIMENT["network_combo"]} ({EXPERIMENT["n_rois"]} ROIs)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(run_artifact_dir, 'loss_curves.png'), dpi=150)
plt.show()
print(f'Final train loss: {history["train_loss"][-1]:.4f}')
print(f'Final val loss:   {history["val_loss"][-1]:.4f}')

In [ ]:
try:
    wandb.finish()
except Exception:
    pass
print(f'Done. Checkpoint: {run_artifact_dir}')